In [3]:
# ==========================
# 0. SETUP & DATA LOADING
# ==========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load data
df = pd.read_csv("cleaned_reviews_with_sentiment.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(3)[["review/comments", "Sentiment"]])

# Keep only relevant columns
TEXT_COL = "review/comments"
LABEL_COL = "Sentiment"

df = df[[TEXT_COL, LABEL_COL]].dropna()
df = df[df[TEXT_COL].str.strip() != ""]  # drop empty texts

# Encode labels -> integers
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df[LABEL_COL])
print("Label classes:", list(label_encoder.classes_))

# Train / test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    df[TEXT_COL].values,
    df["label_id"].values,
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)
print("Train size:", len(X_train), "Test size:", len(X_test))


Shape: (24419, 6)
Columns: ['productUrl', 'review/author/name', 'review/comments', 'review/product_name', 'review/rating', 'Sentiment']
                                     review/comments Sentiment
0  Paheli bar meesho se kuch açha aya hai thanks ...  Positive
1  Lehenga is good but dupatta colour does not ma...   Neutral
2                               Sooo much nice stuff  Positive
Label classes: ['Negative', 'Neutral', 'Positive']
Train size: 19535 Test size: 4884


In [2]:
# ==========================
# 1. CLASSICAL ML MODELS
# ==========================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Common vectorizer
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),  # unigrams + bigrams
    stop_words="english"
)

def evaluate_model(name, model):
    """Train, predict, and print metrics."""
    print(f"\n===== {name} =====")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(
        y_test, y_pred, target_names=label_encoder.classes_
    ))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    return acc

# Pipelines
svm_clf = Pipeline([
    ("tfidf", tfidf),
    ("clf", LinearSVC())
])

nb_clf = Pipeline([
    ("tfidf", tfidf),
    ("clf", MultinomialNB())
])

dt_clf = Pipeline([
    ("tfidf", tfidf),
    ("clf", DecisionTreeClassifier(
        max_depth=20,
        random_state=42
    ))
])

acc_svm = evaluate_model("SVM (LinearSVC)", svm_clf)
acc_nb  = evaluate_model("Multinomial Naive Bayes", nb_clf)
acc_dt  = evaluate_model("Decision Tree", dt_clf)

print("\nModel Accuracies:")
print("SVM :", acc_svm)
print("NB  :", acc_nb)
print("DT  :", acc_dt)



===== SVM (LinearSVC) =====
Accuracy: 0.6112

Classification Report:
              precision    recall  f1-score   support

    Negative       0.44      0.23      0.30      1310
     Neutral       0.49      0.10      0.16       610
    Positive       0.64      0.89      0.74      2964

    accuracy                           0.61      4884
   macro avg       0.53      0.40      0.40      4884
weighted avg       0.57      0.61      0.55      4884

Confusion Matrix:
[[ 301   33  976]
 [  69   59  482]
 [ 311   28 2625]]

===== Multinomial Naive Bayes =====
Accuracy: 0.6368

Classification Report:
              precision    recall  f1-score   support

    Negative       0.85      0.10      0.19      1310
     Neutral       0.93      0.02      0.04       610
    Positive       0.63      1.00      0.77      2964

    accuracy                           0.64      4884
   macro avg       0.80      0.38      0.33      4884
weighted avg       0.73      0.64      0.52      4884

Confusion Matrix:

In [ ]:
# ...existing code...
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Common vectorizer (some params will be tuned)
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),  # default; will be tuned
    stop_words="english"
)

def evaluate_model(name, model):
    # ...existing code...
    print(f"\n===== {name} =====")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(
        y_test, y_pred, target_names=label_encoder.classes_
    ))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    return acc

# Pipelines
svm_pipe = Pipeline([
    ("tfidf", tfidf),
    ("clf", LinearSVC())
])

nb_pipe = Pipeline([
    ("tfidf", tfidf),
    ("clf", MultinomialNB())
])

dt_pipe = Pipeline([
    ("tfidf", tfidf),
    ("clf", DecisionTreeClassifier(random_state=42))
])

# Parameter grids
svm_param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 40000],
    "clf__C": [0.5, 1.0, 2.0],
    "clf__loss": ["hinge", "squared_hinge"]
}

nb_param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 40000],
    "clf__alpha": [0.5, 1.0],
    "clf__fit_prior": [True, False]
}

dt_param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 40000],
    "clf__max_depth": [10, 20, None],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 5]
}

# Grid search (5-fold CV)
svm_grid = GridSearchCV(svm_pipe, svm_param_grid, cv=5, n_jobs=-1, scoring="accuracy", verbose=1)
nb_grid  = GridSearchCV(nb_pipe,  nb_param_grid,  cv=5, n_jobs=-1, scoring="accuracy", verbose=1)
dt_grid  = GridSearchCV(dt_pipe,  dt_param_grid,  cv=5, n_jobs=-1, scoring="accuracy", verbose=1)

print("\nRunning Grid Search for SVM...")
svm_grid.fit(X_train, y_train)
print("Best SVM params:", svm_grid.best_params_)
print("Best SVM CV accuracy:", f"{svm_grid.best_score_:.4f}")
acc_svm = evaluate_model("SVM (GridSearch best)", svm_grid.best_estimator_)

print("\nRunning Grid Search for Naive Bayes...")
nb_grid.fit(X_train, y_train)
print("Best NB params:", nb_grid.best_params_)
print("Best NB CV accuracy:", f"{nb_grid.best_score_:.4f}")
acc_nb = evaluate_model("Naive Bayes (GridSearch best)", nb_grid.best_estimator_)

print("\nRunning Grid Search for Decision Tree...")
dt_grid.fit(X_train, y_train)
print("Best DT params:", dt_grid.best_params_)
print("Best DT CV accuracy:", f"{dt_grid.best_score_:.4f}")
acc_dt = evaluate_model("Decision Tree (GridSearch best)", dt_grid.best_estimator_)

print("\nModel Accuracies (Test Set):")
print("SVM :", acc_svm)
print("NB  :", acc_nb)
print("DT  :", acc_dt)
# ...existing code...


Running Grid Search for SVM...
Fitting 5 folds for each of 36 candidates, totalling 180 fits


c:\Users\MWA3KOR\bosch_genai\lib\site-packages\sklearn\svm\_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Best SVM params: {'clf__C': 1.0, 'clf__loss': 'hinge', 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}
Best SVM CV accuracy: 0.6439

===== SVM (GridSearch best) =====


c:\Users\MWA3KOR\bosch_genai\lib\site-packages\sklearn\svm\_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Accuracy: 0.6456

Classification Report:
              precision    recall  f1-score   support

    Negative       0.83      0.13      0.22      1310
     Neutral       0.72      0.06      0.11       610
    Positive       0.64      1.00      0.78      2964

    accuracy                           0.65      4884
   macro avg       0.73      0.39      0.37      4884
weighted avg       0.70      0.65      0.54      4884

Confusion Matrix:
[[ 167   12 1131]
 [  21   36  553]
 [  12    2 2950]]

Running Grid Search for Naive Bayes...
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best NB params: {'clf__alpha': 0.5, 'clf__fit_prior': True, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}
Best NB CV accuracy: 0.6399

===== Naive Bayes (GridSearch best) =====
Accuracy: 0.6427

Classification Report:
              precision    recall  f1-score   support

    Negative       0.87      0.11      0.20      1310
     Neutral       0.83      0.05      0.09       610
    Posi

: 

In [4]:
# ==========================
# 1. PARAMETERS
# ==========================
import re
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

MAX_VOCAB = 20000
MAX_LEN = 100
EMBED_DIM = 200
BATCH_SIZE = 32
EPOCHS = 20  # increase for better performance
# ==========================
# FIX FOR: NameError: X_train not defined
# ==========================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. LOAD YOUR DATASET
df = pd.read_csv("cleaned_reviews_with_sentiment.csv")

# 2. SELECT TEXT & LABEL COLUMNS
TEXT_COL = "review/comments"
LABEL_COL = "Sentiment"

df = df[[TEXT_COL, LABEL_COL]].dropna()
df = df[df[TEXT_COL].str.strip() != ""]

# 3. ENCODE SENTIMENT LABELS INTO NUMBERS
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df[LABEL_COL])

# 4. TRAIN–TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    df[TEXT_COL].values,
    df["label_id"].values,
    test_size=0.1,
    random_state=42,
    stratify=df["label_id"]
)

print("✅ X_train shape:", X_train.shape)
print("✅ X_test shape :", X_test.shape)
print("✅ y_train shape:", y_train.shape)
print("✅ y_test shape :", y_test.shape)
print("✅ Classes:", label_encoder.classes_)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==========================
# 2. SIMPLE TOKENIZER + VOCAB (NO TENSORFLOW)
# ==========================

def basic_tokenize(text):
    """Very simple tokenizer: lowercase + split on non-letters."""
    text = str(text).lower()
    # keep letters and numbers, split on anything else
    tokens = re.findall(r"[a-z0-9]+", text)
    return tokens

# Build vocabulary from X_train
token_counts = Counter()

for txt in X_train:
    tokens = basic_tokenize(txt)
    token_counts.update(tokens)

# Reserve:
# 0 -> PAD, 1 -> OOV (out-of-vocab)
most_common = token_counts.most_common(MAX_VOCAB - 2)
vocab = {"<PAD>": 0, "<OOV>": 1}
for i, (tok, _) in enumerate(most_common, start=2):
    vocab[tok] = i

vocab_size = len(vocab)
print("Vocab size:", vocab_size)

def text_to_sequence(text, vocab, max_len=MAX_LEN):
    tokens = basic_tokenize(text)
    seq = [vocab.get(tok, 1) for tok in tokens]  # 1 = <OOV>
    # Truncate
    seq = seq[:max_len]
    # Pad
    if len(seq) < max_len:
        seq = seq + [0] * (max_len - len(seq))   # 0 = <PAD>
    return seq

# Convert all X_train, X_test to sequences
X_train_seq = np.array([text_to_sequence(t, vocab, MAX_LEN) for t in X_train], dtype=np.int64)
X_test_seq  = np.array([text_to_sequence(t, vocab, MAX_LEN) for t in X_test],  dtype=np.int64)

num_classes = len(label_encoder.classes_)
print("Train shape:", X_train_seq.shape)
print("Test shape :", X_test_seq.shape)
print("Number of classes:", num_classes)

# ==========================
# 3. CONVERT TO PYTORCH TENSORS + DATALOADERS
# ==========================

X_train_tensor = torch.tensor(X_train_seq, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_seq, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

# ==========================
# 4. LSTM MODEL (PYTORCH)
# ==========================

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        # x: (batch_size, max_len)
        x = self.embedding(x)          # (batch, max_len, embed_dim)
        out, _ = self.lstm(x)          # out: (batch, max_len, hidden_dim)
        last_hidden = out[:, -1, :]    # take last time step
        x = self.dropout(last_hidden)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.fc2(x)           # (batch, num_classes)
        return logits

HIDDEN_DIM = 128

model = LSTMClassifier(
    vocab_size=vocab_size,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=num_classes
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(model)

# ==========================
# 5. TRAINING LOOP
# ==========================

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_x)              # (batch, num_classes)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(batch_y.detach().cpu().numpy())

    epoch_loss = total_loss / len(train_dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {epoch_loss:.4f} - Train Acc: {epoch_acc:.4f}")

# ==========================
# 6. EVALUATION
# ==========================

model.eval()
all_test_preds = []
all_test_labels = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        outputs = model(batch_x)
        preds = torch.argmax(outputs, dim=1)

        all_test_preds.extend(preds.detach().cpu().numpy())
        all_test_labels.extend(batch_y.detach().cpu().numpy())

y_pred_rnn = np.array(all_test_preds)
y_true = np.array(all_test_labels)

print("\n===== RNN (LSTM) Results - PyTorch (No TF) =====")
print("Accuracy:", accuracy_score(y_true, y_pred_rnn))

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred_rnn, target_names=label_encoder.classes_
))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred_rnn))


✅ X_train shape: (21977,)
✅ X_test shape : (2442,)
✅ y_train shape: (21977,)
✅ y_test shape : (2442,)
✅ Classes: ['Negative' 'Neutral' 'Positive']
Using device: cuda
Vocab size: 2194
Train shape: (21977, 100)
Test shape : (2442, 100)
Number of classes: 3
LSTMClassifier(
  (embedding): Embedding(2194, 200, padding_idx=0)
  (lstm): LSTM(200, 128, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=3, bias=True)
)
Epoch [1/20] - Loss: 0.9259 - Train Acc: 0.6065
Epoch [2/20] - Loss: 0.9202 - Train Acc: 0.6069
Epoch [3/20] - Loss: 0.9195 - Train Acc: 0.6069
Epoch [4/20] - Loss: 0.9191 - Train Acc: 0.6069
Epoch [5/20] - Loss: 0.9185 - Train Acc: 0.6069
Epoch [6/20] - Loss: 0.9182 - Train Acc: 0.6069
Epoch [7/20] - Loss: 0.9181 - Train Acc: 0.6069
Epoch [8/20] - Loss: 0.9177 - Train Acc: 0.6069
Epoch [9/20] - Loss: 0.9171 - Train Acc: 0.6069
Epoch [10/20] - Loss:

c:\Users\MWA3KOR\bosch_genai\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\MWA3KOR\bosch_genai\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\MWA3KOR\bosch_genai\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [6]:
y_pred_rnn

array([2, 2, 2, ..., 2, 2, 2], dtype=int64)

In [10]:
from transformers import AutoModelForSequenceClassification
# Download the model from huggingface.co/models
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=3, label2id={}, id2label={},
)

config.json: 0.00B [00:00, ?B/s]

ValueError: The checkpoint you are trying to load has model type `modernbert` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

In [15]:
! pip install -U transformers>=4.48.0


^C


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unstructured-inference 0.7.31 requires opencv-python!=4.7.0.68, which is not installed.
flashrank 0.2.6 requires llama-cpp-python==0.2.76, but you have llama-cpp-python 0.2.82 which is incompatible.
datasets 2.20.0 requires dill<0.3.9,>=0.3.0, but you have dill 0.3.9 which is incompatible.

[notice] A new release of pip is available: 24.1.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ...existing code...
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Use a single consistent model id
MODEL_ID = "answerdotai/ModernBERT-base" # "argilla/ModernBERT-domain-classifier"  # or "answerdotai/ModernBERT-base" consistently everywhere

# Load tokenizer and model consistently
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)

# Optional: build proper label maps from the model config if needed
id2label = model.config.id2label
label2id = model.config.label2id

# Inference pipeline (use device_map="auto" for automatic CPU/GPU selection)
classifier = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

print(classifier("The product is great"))
# ...existing code...

Disabling PyTorch because PyTorch >= 2.1 is required but found 2.0.1+cu117
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
c:\Users\MWA3KOR\bosch_genai\lib\site-packages\torchvision\datapoints\__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transfo

ImportError: 
AutoModelForSequenceClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


In [1]:
# ! pip install sentencepiece --only-binary=:all:

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# 1. LOAD YOUR DATASET
df = pd.read_csv("cleaned_reviews_with_sentiment.csv")

# 2. SELECT TEXT & LABEL COLUMNS
TEXT_COL = "review/comments"
LABEL_COL = "Sentiment"

df = df[[TEXT_COL, LABEL_COL]].dropna()
df = df[df[TEXT_COL].str.strip() != ""]

# 3. ENCODE SENTIMENT LABELS INTO NUMBERS
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df[LABEL_COL])

# 4. TRAIN–TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    df[TEXT_COL].values,
    df["label_id"].values,
    test_size=0.1,
    random_state=42,
    stratify=df["label_id"]
)

print("✅ X_train shape:", X_train.shape)
print("✅ X_test shape :", X_test.shape)
print("✅ y_train shape:", y_train.shape)
print("✅ y_test shape :", y_test.shape)
print("✅ Classes:", label_encoder.classes_)

import torch 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
import numpy as np

BASE_MODEL = "microsoft/Multilingual-MiniLM-L12-H384"  # multilingual MiniLM

# Build HF Datasets from your existing splits
train_df = pd.DataFrame({ "text": X_train, "label": y_train })
test_df  = pd.DataFrame({ "text": X_test,  "label": y_test  })

hf_train = Dataset.from_pandas(train_df)
hf_test  = Dataset.from_pandas(test_df)

# Tokenizer
mini_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)

def tokenize_fn(batch):
    return mini_tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=256
    )

hf_train_tok = hf_train.map(tokenize_fn, batched=True)
hf_test_tok  = hf_test.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=mini_tokenizer)

# Label maps from your label_encoder (ensure exactly 3 classes)
num_labels = 3
id2label = {i: l for i, l in enumerate(label_encoder.classes_)}
label2id = {l: i for i, l in enumerate(label_encoder.classes_)}

# Model for 3 classes
mini_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Training args (adjust batch sizes/epochs to your hardware)
# training_args = TrainingArguments(
#     output_dir="./miniLM_sentiment",
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     learning_rate=3e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=32,
#     num_train_epochs=3,
#     weight_decay=0.01,
#     logging_steps=50,
#     load_best_model_at_end=True,
#     metric_for_best_model="accuracy",
#     report_to="none"
# )
training_args = TrainingArguments(
output_dir="./miniLM_sentiment",
save_strategy="epoch",
learning_rate=3e-5,
per_device_train_batch_size=16,
per_device_eval_batch_size=32,
num_train_epochs=3,
weight_decay=0.01,
logging_steps=50,
load_best_model_at_end=False,
report_to="none"
)
mini_model.train()
metrics = mini_model.evaluate(hf_test_tok)
print("========== Metrics ",metrics)
# Metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    return {
        "accuracy": acc,
        "precision_macro": prec,
        "recall_macro": rec,
        "f1_macro": f1
    }

trainer = Trainer(
    model=mini_model,
    args=training_args,
    train_dataset=hf_train_tok,
    eval_dataset=hf_test_tok,
    tokenizer=mini_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# Evaluate
metrics = trainer.evaluate(hf_test_tok)
print("\n===== MiniLM (Multilingual) Results =====")
print(metrics)

# Predictions + report
pred_logits = trainer.predict(hf_test_tok).predictions
y_pred_minilm = np.argmax(pred_logits, axis=1)
print("\nClassification Report (MiniLM):")
print(classification_report(
    y_test, y_pred_minilm, target_names=label_encoder.classes_
))
print("Confusion Matrix (MiniLM):")
print(confusion_matrix(y_test, y_pred_minilm))

✅ X_train shape: (21977,)
✅ X_test shape : (2442,)
✅ y_train shape: (21977,)
✅ y_test shape : (2442,)
✅ Classes: ['Negative' 'Neutral' 'Positive']
Using device: cuda


Map:   0%|          | 0/21977 [00:00<?, ? examples/s]

Map:   0%|          | 0/2442 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/Multilingual-MiniLM-L12-H384 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
# ==========================
# 3. BERT-BASED CLASSIFIER
# ==========================
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    DataCollatorWithPadding,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import torch

MODEL_NAME = "distilbert-base-uncased"  # small, good starting point

# Build Hugging Face Dataset
train_df = pd.DataFrame({ "text": X_train, "label": y_train })
test_df  = pd.DataFrame({ "text": X_test,  "label": y_test  })

hf_train = Dataset.from_pandas(train_df)
hf_test  = Dataset.from_pandas(test_df)

# Tokenizer
tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer_bert(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=128
    )

hf_train_tok = hf_train.map(tokenize_fn, batched=True)
hf_test_tok  = hf_test.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer_bert)

# Load model
id2label = {i: l for i, l in enumerate(label_encoder.classes_)}
label2id = {l: i for i, l in enumerate(label_encoder.classes_)}

model_bert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_encoder.classes_),
    id2label=id2label,
    label2id=label2id
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./bert_sentiment",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

# Metrics function
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    return {
        "accuracy": acc,
        "precision_macro": prec,
        "recall_macro": rec,
        "f1_macro": f1
    }

trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=hf_train_tok,
    eval_dataset=hf_test_tok,
    tokenizer=tokenizer_bert,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# Evaluate on test set
metrics = trainer.evaluate(hf_test_tok)
print("\n===== BERT (DistilBERT) Results =====")
print(metrics)


In [ ]:
pred_logits = trainer.predict(hf_test_tok).predictions
y_pred_bert = np.argmax(pred_logits, axis=1)

print("\nClassification Report (BERT):")
print(classification_report(
    y_test, y_pred_bert, target_names=label_encoder.classes_
))
print("Confusion Matrix (BERT):")
print(confusion_matrix(y_test, y_pred_bert))
